# Colab — End-to-End Fine-Tune & Predict
**Multilingual Health QA (HASH / Zindi).** Reproducible run: install → train seq2seq →
evaluate on Val (leaderboard-mirroring ROUGE) → generate a **reviewable** Test prediction
CSV. Maps to rubric crit. 1, 2, 9 (leaderboard, tracking, reproducibility).

> ⚠️ This notebook does **not** upload anything. It writes a local submission CSV for you
> to review and submit manually.

## 0. Runtime check (use a GPU runtime)
**Colab:** Runtime → Change runtime type → **T4**.  **Kaggle:** Settings → Accelerator → **GPU T4 ×2** or **P100**.
fp32 training (below) is stable on any of these — no bf16 needed.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. Get the code + data

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
REPO_URL = 'https://github.com/NzizaPacifique250/multilingual_health_project.git'
if IN_COLAB or IN_KAGGLE:
    base = '/content' if IN_COLAB else '/kaggle/working'
    path = os.path.join(base, 'multilingual_health_project')   # absolute -> never nests
    if not os.path.isdir(path):
        !git clone -q {REPO_URL} {path}
    %cd {path}
    !git pull -q          # always run the latest committed code
    !pip install -q -r requirements.txt
sys.path.insert(0, os.getcwd())
# The data CSVs are tracked in the repo, so the clone already includes them.
assert os.path.exists('datasets/Train.csv'), 'place data CSVs in datasets/'
print('working dir:', os.getcwd())

## 2. Train — run **B2s** (mt5-small, full fine-tune, T4-safe)
T4 only supports **fp16**, and mT5 + fp16 diverges to `NaN` (likely hurt B1). So we train in
**fp32** (stable). mt5-small is small enough to **full fine-tune** in fp32 on a T4 (LoRA would
just underfit it), so we drop LoRA here and keep the rest of the B2 recipe:
- `--epochs 5`, `--upsample_low_resource 3.0` (Amh/Swa/Lug/Aka ×3 → ~53k rows/epoch)
- `--num_beams 5 --length_penalty 1.3 --gen_min_len 16` (recall-oriented decoding)
- **no `--fp16/--bf16`** → fp32. ~2-4h on a T4.

> This is the fast, stable T4 path. mt5-small has a lower ceiling than mt5-base — if it
> trains cleanly and beats B1, the next move is NLLB-600M (fp16-stable, better African
> coverage) for a real shot at the 0.39 retrieval floor.

In [ ]:
# B2s: mt5-small FULL fine-tune, fp32, T4-safe.
# fp32 avoids mT5's fp16 NaN; gradient checkpointing + train_bs=4 fit the T4's ~14.5GB.
# ~4-5h for 3 epochs. Saves a checkpoint each epoch, so a disconnect is recoverable.
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m src.train \
    --model_name google/mt5-small \
    --output_dir outputs/B2_mt5small \
    --epochs 3 --train_bs 4 --eval_bs 4 --grad_accum 4 \
    --lr 5e-4 --max_input_len 128 --max_target_len 256 \
    --upsample_low_resource 3.0 \
    --num_beams 5 --gen_max_len 256 --gen_min_len 16 \
    --no_repeat_ngram 3 --length_penalty 1.3 \
    --eval_subsample 1000 --gradient_checkpointing

## 3. Evaluate on Val (full per-language ROUGE)

In [ ]:
!python -m src.infer --model_dir outputs/B2_mt5small --split val \
    --num_beams 5 --gen_max_len 256 --gen_min_len 16 \
    --no_repeat_ngram 3 --length_penalty 1.3 --tag B2_mt5small

## 4. Generate Test predictions → reviewable submission CSV (no upload)

In [ ]:
!python -m src.infer --model_dir outputs/B2_mt5small --split test \
    --num_beams 5 --gen_max_len 256 --gen_min_len 16 \
    --no_repeat_ngram 3 --length_penalty 1.3 --tag B2_mt5small

In [ ]:
import pandas as pd
sub = pd.read_csv('outputs/submission_B2_mt5small.csv')
print(sub.shape); sub.head()  # REVIEW before uploading to Zindi

## 5. Manual review checklist (do this before submitting)
- [ ] Spot-check Amharic rows are **Ge'ez script** (infer.py prints a wrong-language count).
- [ ] No empty / single-token answers; lengths look reasonable per language.
- [ ] Val proxy **beats the TF-IDF B0 (0.394)** and prior runs (`experiments/results.csv`).
- [ ] IDs match `SampleSubmission.csv` (submit.py validates this).
Then upload `outputs/submission_*.csv` to Zindi yourself and record the LB score in the log.

## 6. Log the run
Append a row to `experiments/results.csv` and a note to `experiments/log.md`:
config (model, epochs, lr, decoding), Val R1/RL/proxy, LB score, and one-line takeaway.